# 🎙️ Pipeline Skripsi VoxCPM & Multi-Generator: Residual & Modulasi Speech Deepfake

Jalankan cell di bawah ini **secara berurutan (Shift + Enter)** dari atas ke bawah.

### 🌟 Fitur Pipeline (Full Google Drive Dataset):
1. **Full Dataset dari Google Drive (`Folder_data_inti`)**: Memindai folder `Voxcpm`, `OpenVoice`, `F5TTS`, `E2TTS`, `Suara_real`, dsb murni dari Drive Anda.
2. **Inspector Susunan Folder Drive**: Cell khusus untuk melihat struktur folder & nama file audio di Google Drive Anda.
3. **Pemisahan Subfolder `training` vs `testing`**: Data di subfolder `training` digunakan untuk pelatihan model (split 70% Train, 15% Val, 15% Test dengan *data shuffling* dan *speaker split*). Data di subfolder `testing` digunakan khusus sebagai **Dedicated Cross-Generator Blind Test**.
4. **CSV Prediction Mapping Detail**: Menghasilkan pemetaan per file audio (`file_name`, `generator_source`, `y_true`, `y_pred`, `y_score`, `is_correct`).
5. **Visualisasi Confusion Matrix & Metrik Per-Generator**: Menyimpan dan menampilkan matriks kebingungan (PNG & CSV) serta tabel komparasi per generator TTS.

## 1. Setup Environment & Clone Repository

In [ ]:
!git clone https://github.com/alvinrw/skripsi_fase2.git
%cd skripsi_fase2

!pip install -q numpy pandas scipy scikit-learn librosa soundfile xgboost pyyaml joblib tqdm statsmodels matplotlib seaborn
print("✅ Setup environment selesai!")

## 2. Hubungkan Google Drive (Wajib!)
Ini memastikan model dan hasil perhitungan (file CSV & PNG) **tidak hilang** saat Colab ditutup, dan membaca dataset mentah murni dari folder Drive Anda.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/skripsi_results'
for folder in ['results', 'checkpoints', 'manifests', 'figures']:
    os.makedirs(f'{DRIVE_PATH}/{folder}', exist_ok=True)
    if not os.path.islink(folder):
        if os.path.exists(folder):
            !rm -rf {folder}
        os.symlink(f'{DRIVE_PATH}/{folder}', folder)

print(f"✅ Drive terhubung! Semua hasil akan otomatis tersimpan di: {DRIVE_PATH}")

## 2.5. Inspeksi Susunan Folder Google Drive 📂🔍
Cell ini akan menampilkan struktur hierarki folder dan jumlah file audio di dalam folder Google Drive Anda (`Folder_data_inti`) agar Anda bisa memastikan posisi folder `training` dan `testing` sudah pas.

In [ ]:
import os

DRIVE_DATA_DIR = "/content/drive/MyDrive/Folder_data_inti"
if not os.path.exists(DRIVE_DATA_DIR):
    DRIVE_DATA_DIR = "/content/drive/MyDrive/VoxCPM"

print(f"📁 SUSUNAN DIREKTORI GOOGLE DRIVE:")
print(f"Path: {DRIVE_DATA_DIR}\n" + "="*60)

if os.path.exists(DRIVE_DATA_DIR):
    audio_exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
    for root, dirs, files in os.walk(DRIVE_DATA_DIR):
        level = root.replace(DRIVE_DATA_DIR, '').count(os.sep)
        indent = ' ' * 4 * level
        folder_name = os.path.basename(root) if os.path.basename(root) else DRIVE_DATA_DIR
        audio_files = [f for f in files if os.path.splitext(f)[1].lower() in audio_exts]
        print(f"{indent}📂 {folder_name}/ ({len(audio_files)} file audio)")
        
        if audio_files:
            for sample in audio_files[:3]:
                print(f"{indent}    📄 {sample}")
            if len(audio_files) > 3:
                print(f"{indent}    ... dan {len(audio_files)-3} file audio lainnya")
else:
    print(f"⚠️ Folder {DRIVE_DATA_DIR} belum ditemukan. Pastikan Google Drive sudah di-mount.")

## 3. Persiapan & QC Dataset (`Folder_data_inti`)
Melakukan scanning folder generator murni dari Google Drive Anda (`Voxcpm`, `OpenVoice`, `F5TTS`, `E2TTS`), mendeteksi subfolder `training` vs `testing`, melakukan QC audio, pemotongan (*chunking*) 2 detik, serta pemisahan manifest.

*(Sesuaikan `DRIVE_DATA_DIR` jika lokasi folder data utama Anda berada di tempat lain)*

In [ ]:
# Tentukan lokasi folder data utama Anda di Google Drive
DRIVE_DATA_DIR = "/content/drive/MyDrive/Folder_data_inti"

if not os.path.exists(DRIVE_DATA_DIR):
    DRIVE_DATA_DIR = "/content/drive/MyDrive/VoxCPM"

print(f"📂 Menggunakan lokasi dataset: {DRIVE_DATA_DIR}")

# Jalankan persiapan dataset murni dari Google Drive
!python src/run_pipeline.py --steps prepare --drive_dir "{DRIVE_DATA_DIR}" --out_dir "/content/VoxCPM_processed"
print("✅ Persiapan dataset selesai!")

## 4. Eksekusi Penuh Pipeline Skripsi 🔥
Ekstraksi fitur MFCC/LFCC/Residual/Modulasi, pelatihan model Machine Learning pada dataset *training* (dengan *data shuffling*), serta pengujian pada **In-Domain Test Set** dan **Dedicated Separate Testing Set**.

> ⏳ **Catatan:** Ekstraksi fitur dan *training* akan memakan waktu tergantung jumlah file audio.

In [ ]:
!python src/run_pipeline.py --steps features --duration 2s
!python src/run_pipeline.py --steps train --duration 2s
!python src/run_pipeline.py --steps stats consistency bootstrap evaluate --duration 2s
print('✅ Eksekusi pipeline durasi 2s selesai!')

## 5. Inspeksi CSV Prediction Mapping & Confusion Matrices 📊
Menampilkan preview hasil klasifikasi per file audio, per-generator breakdown (Voxcpm vs OpenVoice vs F5TTS vs E2TTS), serta gambar Confusion Matrix.

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# 1. Ringkasan Metrik Pengujian
metrics_path = 'results/metrics_2s.csv'
if os.path.exists(metrics_path):
    df_m = pd.read_csv(metrics_path)
    print("🏆 PERBANDINGAN PERFORMA MODEL (Validation vs Test vs Separate Test):")
    display(df_m[['model', 'split', 'auc', 'eer', 'accuracy', 'f1_macro']])
else:
    print("⚠️ File metrics belum ditemukan.")

# 2. Preview CSV Prediction Mapping Terpisah (Separate Blind Test)
mapping_files = sorted(glob.glob('results/prediction_mapping_*_separate_test.csv'))
if mapping_files:
    print("\n📄 PREVIEW CSV PREDICTION MAPPING (Separate Blind Test):")
    df_map = pd.read_csv(mapping_files[0])
    display(df_map.head(10)[['file_name', 'generator_source', 'true_label', 'predicted_label', 'confidence_score_fake', 'is_correct']])
    
# 3. Breakdown Metrics Per Generator TTS
gen_files = sorted(glob.glob('results/per_generator_metrics_*_separate_test.csv'))
if gen_files:
    print("\n📊 BREAKDOWN PERFORMA PER-GENERATOR TTS (Voxcpm vs OpenVoice vs F5TTS vs E2TTS):")
    df_gen = pd.read_csv(gen_files[0])
    display(df_gen)

# 4. Visualisasi Confusion Matrix PNG
cm_pngs = sorted(glob.glob('results/confusion_matrices/*.png'))
if cm_pngs:
    print("\n🖼️ CONFUSION MATRICES (Top 3 Model):")
    for p in cm_pngs[:3]:
        print(f"File: {p}")
        display(Image(filename=p))


## 🎉 Selesai!
Semua proses telah tuntas. Seluruh file **CSV Mapping**, **Per-Generator Breakdown**, **Confusion Matrix PNG & CSV**, serta **Checkpoint Model** telah otomatis tersimpan di Google Drive Anda (`/content/drive/MyDrive/skripsi_results`). Siap dipakai langsung di naskah skripsi Anda!